# XLAV — E2: text prompt và visual prompt (CLIPSeg, PhraseCut miniv)

Câu hỏi: chỉ cho CLIPSeg biết vật cần tìm **bằng chữ** hay **bằng một ảnh mẫu (support)** thì tốt hơn,
và ảnh mẫu nên được xử lý thế nào? Mỗi query miniv được dự đoán với 5 biến thể prompt:

| Biến thể | Prompt |
|---|---|
| `text_phrase` | Cụm từ gốc, ví dụ `red truck` (giống E1, là gốc để so sánh) |
| `text_name` | Chỉ danh từ chính `truck`, cùng mức thông tin với ảnh support |
| `visual_crop` | Ảnh support crop theo bbox vật (nới 10%) |
| `visual_dark` | Cả ảnh support, nền ngoài vật giữ 10% độ sáng |
| `visual_crop_dark` | Kết hợp crop và tối nền |

**Support** lấy từ **ảnh miniv khác** có cùng danh từ chính (`phrase_structure.name`). Mask dùng để crop/tối nền
là GT **của ảnh support**; mask GT của ảnh query không bao giờ được dùng để tạo prompt. Mỗi query dùng đúng
`SUPPORTS_PER_QUERY` support từ các ảnh khác nhau; metric của biến thể visual là trung bình trên các support.
Danh sách cặp lưu ở `e2_pairs.json`.

1. Bật **GPU T4 x2** và **Internet** trong Settings của notebook Kaggle.
2. **Run All**. Kết quả ở `/kaggle/working/clipseg_e2_text_vs_visual`, kèm file ZIP.

Đây là phiên bản rút gọn ý tưởng Bảng 2 của paper (paper đo ΔP trên LVIS, có thêm blur); số liệu không so trực tiếp.


## 1. Dependency

Giữ PyTorch/CUDA, NumPy, Pillow, scikit-learn và matplotlib sẵn có của Kaggle.
Cell này chỉ pin Transformers và gdown; không cài wheel PyTorch CPU.
Nếu kernel đã import Transformers phiên bản khác trước đó, restart session rồi chạy lại từ đầu.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.55.4", "gdown==5.2.0",
])


## 2. Cấu hình

`MAX_QUERIES=30` theo kế hoạch; nếu không đủ query có support hợp lệ, notebook chạy với số có được và ghi rõ N.
`MIN_SUPPORT_BOX_FRACTION` loại support quá nhỏ (tổng diện tích box của vật / diện tích ảnh support),
vì crop từ vật vài pixel không còn là ảnh mẫu có nghĩa.

`DECODER_CHECKPOINT=None` dùng decoder pretrained. Để đánh giá decoder đã train theo `EPOCHS` + `BATCH_SIZE`:
chạy notebook E1 với `TRAIN_DECODER=True`, lưu version để Output thành dataset, Add Input dataset đó vào notebook này,
rồi trỏ `DECODER_CHECKPOINT` tới `checkpoints/decoder_epoch_XXX.pt` của **epoch cuối**.
Kết quả ghi đè thư mục output; tải ZIP của lượt pretrained trước khi chạy lượt có checkpoint.


In [ ]:
from pathlib import Path

MAX_QUERIES = 30                 # 30 = pilot; 8 = smoke; None = mọi query đủ support
SUPPORTS_PER_QUERY = 3           # mỗi query dùng đúng K support từ K ảnh khác nhau
MIN_SUPPORT_BOX_FRACTION = 0.02  # bỏ support có vật quá nhỏ
CROP_PADDING = 0.1               # nới bbox support 10% mỗi phía trước khi crop
DARK_FACTOR = 0.1                # nền giữ 10% độ sáng, như Bảng 2 của paper
SUPPORT_VIEW_MODE = "legacy"    # legacy: ba biến thể cũ | train_matched: visual_prompt giống hệt lúc train
SUPPORT_AUG_CROP = True          # train_matched: crop vuông như train, seed cố định theo support ID
BATCH_SIZE = 9                   # số visual prompt mỗi forward (support × biến thể); giảm nếu thiếu VRAM
OUTPUT_NAME = "clipseg_e2_text_vs_visual"

# None = decoder pretrained. Hoặc decoder đã train trong notebook E1 theo EPOCHS + BATCH_SIZE, ví dụ
# "/kaggle/input/<output-e1>/clipseg_e1/checkpoints/decoder_epoch_004.pt"
DECODER_CHECKPOINT = None
DATA_ROOT = None                 # ví dụ: "/kaggle/input/phrasecut-miniv/VGPhraseCut_v0"
EVAL_SPLIT = "miniv"             # "miniv" | "val" | "test"; support của E2 lấy trong cùng split
MODEL_PATH = None                # thư mục checkpoint offline, hoặc None
IMAGE_ROOT = None                # thư mục chứa ảnh <image_id>.jpg; None = tìm trong mọi Kaggle Input
WORK_ROOT = Path("/kaggle/working")
REQUIRE_GPU = True               # False chỉ khi kiểm tra trên máy CPU
USE_AMP = False                  # mặc định FP32 như E1
SEED = 42
THRESHOLD = 0.3
CALIBRATION_FRACTION = 0.2
THRESHOLD_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
N_BOOTSTRAP = 2000               # bootstrap theo ảnh cho CI 95% của ΔIoU
TIE_TOLERANCE = 0.01             # |ΔIoU| ≤ 0,01 tính là hòa
SAVE_MASKS = True
N_VISUALIZATIONS = 6
DOWNLOAD_WORKERS = 4

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
PHRASECUT_API_COMMIT = "276470b7773d8a230aee5b5536876fbc485fb04b"


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext

import gdown
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor
from IPython.display import display

assert 0 < CALIBRATION_FRACTION < 1
assert 0 <= THRESHOLD <= 1 and all(0 <= t <= 1 for t in THRESHOLD_GRID)
assert N_BOOTSTRAP >= 100 and TIE_TOLERANCE >= 0
random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Bật GPU trong Kaggle Settings rồi restart session.")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        torch.ones(1, device=device).add_(1).item()
    except RuntimeError as exc:
        raise RuntimeError("PyTorch của Kaggle không chạy được trên GPU này (thường gặp với P100). "
                           "Chọn Accelerator GPU T4 x2 rồi restart session.") from exc
assert not USE_AMP or device.type == "cuda", "FP16 AMP yêu cầu CUDA."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT = WORK_ROOT / OUTPUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
# Xóa kết quả lượt cũ để không trộn artifact vào ZIP.
for folder in ["masks", "visualizations"]:
    shutil.rmtree(OUT / folder, ignore_errors=True)
for filename in ["summary.json", "run.json"]:
    (OUT / filename).unlink(missing_ok=True)
(WORK_ROOT / f"{OUTPUT_NAME}.zip").unlink(missing_ok=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "numpy", "Pillow", "scikit-learn", "matplotlib", "gdown"]}
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print(json.dumps(versions, indent=2))


## 3. Lấy annotation chính thức và ghép query với support

Annotation tải như E1. Query hợp lệ khi có ít nhất `SUPPORTS_PER_QUERY` **ảnh khác** chứa task cùng danh từ chính
và đủ lớn. Chọn query và support bằng seed 42, rồi chia calibration/evaluation theo image_id của query.
Ảnh cần tải gồm ảnh query và ảnh support.


In [ ]:
# Drive ID của annotation chính thức, theo download_dataset.py của PhraseCut.
REFER_DRIVE_IDS = {"miniv": "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E", "val": "1UyojArOFPlsSeNbA9fHWjCjOOU-OCohG",
                   "test": "1jrzXm1gcq6f5hNDeamZd0UmyyHUv61IZ", "train": "1qx-0q6r9r0YUGpoyT0B8HJKmUFWQDSu7"}

def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob(f"refer_{EVAL_SPLIT}.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều dataset chứa refer_{EVAL_SPLIT}.json; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_annotations"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / f"refer_{EVAL_SPLIT}.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, REFER_DRIVE_IDS[EVAL_SPLIT])
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == EVAL_SPLIT}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), f"Annotation không thuộc split {EVAL_SPLIT}."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

assert all(str(t.get("phrase_structure", {}).get("name", "")).strip() for t in tasks), "Thiếu phrase_structure.name."
assert SUPPORTS_PER_QUERY >= 1 and (MAX_QUERIES is None or MAX_QUERIES >= 1) and 0 <= MIN_SUPPORT_BOX_FRACTION < 1

def target_key(task):
    return task["phrase_structure"]["name"].strip().lower()

def box_fraction(task):
    info = info_by_id[int(task["image_id"])]
    area = sum(max(w, 0) * max(h, 0) for _, _, w, h in task["instance_boxes"])
    return area / (int(info["width"]) * int(info["height"]))

ordered = sorted(tasks, key=lambda t: str(t["task_id"]))
supports_by_key = defaultdict(dict)   # key -> image_id -> một task support (task_id nhỏ nhất)
for task in ordered:
    if box_fraction(task) >= MIN_SUPPORT_BOX_FRACTION:
        supports_by_key[target_key(task)].setdefault(int(task["image_id"]), task)
eligible = []
for task in ordered:
    candidates = [s for image_id, s in sorted(supports_by_key[target_key(task)].items())
                  if image_id != int(task["image_id"])]
    if len(candidates) >= SUPPORTS_PER_QUERY:
        eligible.append((task, candidates))
rng = random.Random(SEED)
chosen = eligible.copy() if MAX_QUERIES is None else rng.sample(eligible, min(MAX_QUERIES, len(eligible)))
chosen.sort(key=lambda pair: (int(pair[0]["image_id"]), str(pair[0]["task_id"])))
pairs = [{"query": task, "supports": rng.sample(candidates, SUPPORTS_PER_QUERY)} for task, candidates in chosen]
if MAX_QUERIES is not None and len(pairs) < MAX_QUERIES:
    print(f"Chỉ có {len(pairs)} query có đủ support hợp lệ (yêu cầu {MAX_QUERIES}); báo cáo đúng N này.")
if len(pairs) < 2:
    raise ValueError("Không đủ query có support hợp lệ; giảm SUPPORTS_PER_QUERY hoặc MIN_SUPPORT_BOX_FRACTION.")
for pair in pairs:
    assert all(int(s["image_id"]) != int(pair["query"]["image_id"]) for s in pair["supports"]), "Support trùng ảnh query."
    assert len({int(s["image_id"]) for s in pair["supports"]}) == SUPPORTS_PER_QUERY

selected = [pair["query"] for pair in pairs]
query_image_ids = sorted({int(t["image_id"]) for t in selected})
if len(query_image_ids) < 2:
    raise ValueError("Cần ít nhất 2 ảnh query để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = query_image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(query_image_ids) - 1, max(1, round(len(query_image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in query_image_ids}
image_ids = sorted(set(query_image_ids) | {int(s["image_id"]) for p in pairs for s in p["supports"]})
write_json(OUT / "e2_pairs.json", [{"task_id": str(p["query"]["task_id"]), "image_id": int(p["query"]["image_id"]),
    "phrase": p["query"]["phrase"], "target_key": target_key(p["query"]),
    "group": group_for_image[int(p["query"]["image_id"])],
    "supports": [{"task_id": str(s["task_id"]), "image_id": int(s["image_id"]), "phrase": s["phrase"],
                  "box_fraction": round(box_fraction(s), 4)} for s in p["supports"]]} for p in pairs])
print(f"Annotation: {len(tasks)} queries / {len(info_by_id)} miniv images; eligible queries: {len(eligible)}")
print(f"Selected: {len(pairs)} queries / {len(query_image_ids)} query images; calibration {n_cal} images")
print(f"Images to load (query + support): {len(image_ids)}")


## 4. Tải và kiểm tra ảnh

Ưu tiên ảnh đã có trong dataset input. Ảnh thiếu sẽ tải từ URL Visual Genome trong metadata
bằng HTTPS vào working cache, retry có giới hạn. Lỗi tải hoặc kích thước khác metadata làm
dừng lượt chạy: không âm thầm bỏ query để tạo số liệu đẹp hơn.


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
def image_search_roots(*extra):
    """Nơi tìm ảnh <image_id>.jpg: IMAGE_ROOT nếu đặt, nếu không thì mọi Kaggle Input; cộng thư mục annotation."""
    roots = [Path(IMAGE_ROOT)] if IMAGE_ROOT else ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    roots += [Path(r) for r in extra]
    # Bỏ thư mục nằm trong thư mục đã quét để không quét hai lần.
    return [r for i, r in enumerate(roots) if r.is_dir() and not any(r == q or q in r.parents for q in roots[:i])]

def index_local_images(roots, wanted_ids):
    index = {}
    for search_root in roots:
        for path in sorted(search_root.rglob("*")):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.stem.isdigit() and int(path.stem) in wanted_ids:
                index.setdefault(int(path.stem), path)
    return index

local_images = index_local_images(image_search_roots(data_root), set(image_ids))
print(f"Ảnh có sẵn trong Input: {len(local_images)}/{len(image_ids)}; phần còn lại tải từ Visual Genome")

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


## 5. Nạp model, inference và metric

Nếu `DECODER_CHECKPOINT` được đặt, decoder được thay bằng decoder đã train trong notebook E1
(CLIP vẫn là backbone đóng băng của checkpoint gốc). Epoch, batch size và lịch sử loss đọc từ checkpoint
được ghi vào `summary.json` và `run.json`. E1 chỉ train trên split `train`, nên miniv vẫn là dữ liệu chưa thấy.

Model, tiền xử lý ảnh (352×352 như CLIPSeg gốc), rasterize GT và metric giống hệt E1, gồm cả bộ metric
theo protocol paper (`paper_protocol`: miou_0.3, fgiou_0.3, AP ở 352×352). `predict` nhận **hoặc** danh sách text prompt
**hoặc** danh sách ảnh support (truyền vào `conditional_pixel_values` của CLIPSeg),
luôn dự đoán trên cùng một ảnh query. Logits nội suy bilinear về kích thước gốc rồi sigmoid.

`paired_comparison` so từng query với biến thể gốc: ΔIoU = biến thể − gốc, CI 95% bằng
bootstrap theo **ảnh** (các query cùng ảnh không độc lập), kèm số query thắng/hòa/thua.


In [ ]:
load_kwargs = {"local_files_only": True} if MODEL_PATH else {"revision": MODEL_REVISION}
model_source = str(MODEL_PATH) if MODEL_PATH else MODEL_ID
processor = CLIPSegProcessor.from_pretrained(model_source, use_fast=False, **load_kwargs)
model = CLIPSegForImageSegmentation.from_pretrained(model_source, **load_kwargs).to(device).eval()
assert processor.image_processor.size == {"height": 352, "width": 352}, "Processor không có input 352 như baseline."
assert model.config.vision_config.patch_size == 16, "Baseline yêu cầu ViT-B/16."
print("Model:", model_source, "| parameters:", sum(p.numel() for p in model.parameters()))

try:
    from skimage.draw import polygon2mask
except ImportError as exc:
    raise ImportError("Cần scikit-image (có sẵn trên Kaggle); cài bằng pip install scikit-image.") from exc
try:
    from scipy.integrate import simpson
except ImportError:  # SciPy cũ
    from scipy.integrate import simps as simpson

# Tiền xử lý và độ phân giải chấm điểm theo score.py + datasets/phrasecut.py của CLIPSeg.
PAPER_SIZE = 352
PAPER_THRESHOLDS = np.linspace(0, 1, 51)[1:-1]   # 49 ngưỡng 0,02–0,98 như FixedIntervalMetrics
image_mean = torch.tensor(processor.image_processor.image_mean, dtype=torch.float32).view(3, 1, 1)
image_std = torch.tensor(processor.image_processor.image_std, dtype=torch.float32).view(3, 1, 1)

def rasterize_gt(task, width, height, allow_empty=False):
    """Hợp polygon mọi instance bằng skimage polygon2mask (x, y đảo thành hàng, cột), như PhraseCut.load_sample."""
    target = np.zeros((height, width), dtype=bool)
    for instance in task["Polygons"]:
        for polygon in instance:
            if len(polygon) >= 3:
                target |= polygon2mask((height, width), np.asarray(polygon, dtype=np.float64)[:, ::-1])
    if not allow_empty and not target.any():
        raise ValueError(f"GT rỗng: task {task['task_id']}")
    return target

def paper_pixel_values(image):
    """Ảnh RGB → [3,352,352]: resize bilinear align_corners=True trên ảnh gốc, /255, chuẩn hóa mean/std."""
    pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1).unsqueeze(0)
    pixels = F.interpolate(pixels, size=(PAPER_SIZE, PAPER_SIZE), mode="bilinear", align_corners=True)[0]
    return (pixels / 255 - image_mean) / image_std

def paper_target(target):
    """GT ở độ phân giải gốc → 352×352 bằng nearest, như nnf.interpolate trong load_sample."""
    mask = torch.from_numpy(target.astype(np.uint8))[None, None]
    return F.interpolate(mask, size=(PAPER_SIZE, PAPER_SIZE), mode="nearest")[0, 0].numpy().astype(bool)

def paper_threshold_stats(probability, target):
    """tp, fp, fn, tn tại 49 ngưỡng (dương khi p > t) cho một query, như FixedIntervalMetrics.add."""
    order = np.argsort(probability, axis=None, kind="stable")
    p, g = probability.ravel()[order], target.ravel()[order]
    below = np.searchsorted(p, PAPER_THRESHOLDS, side="right")   # số pixel p <= t
    fn = np.concatenate([[0], np.cumsum(g, dtype=np.int64)])[below]
    tp = int(g.sum()) - fn
    fp = (len(p) - below) - tp
    tn = below - fn
    return np.stack([tp, fp, fn, tn]).astype(np.int64)

def paper_metrics(stats):
    """Metric bảng PhraseCut của paper từ stats [Q,4,49]: miou (trung bình theo query), fgiou (cộng dồn), AP Simpson."""
    stats = np.asarray(stats, dtype=np.float64)
    tp, fp, fn = stats[:, 0], stats[:, 1], stats[:, 2]
    s_tp, s_fp, s_fn = tp.sum(0), fp.sum(0), fn.sum(0)
    made = (s_tp + s_fp) > 0
    pairs = sorted(set(zip((s_tp / (1 + s_tp + s_fn))[made], (s_tp / (1 + s_tp + s_fp))[made])))
    recalls, precisions = (np.array(v) for v in zip(*pairs))
    fgiou = s_tp / (1 + s_tp + s_fp + s_fn)
    miou = (tp / (1 + tp + fp + fn)).mean(0)
    index = {t: int(np.argmin(np.abs(PAPER_THRESHOLDS - t))) for t in (0.3, 0.5)}
    return {"n_queries": len(stats), "miou_0.3": float(miou[index[0.3]]), "fgiou_0.3": float(fgiou[index[0.3]]),
            "fgiou_0.5": float(fgiou[index[0.5]]), "ap": float(simpson(precisions, x=recalls)),
            "miou_best": float(miou.max()), "miou_best_t": float(PAPER_THRESHOLDS[miou.argmax()]),
            "fgiou_best": float(fgiou.max()), "fgiou_best_t": float(PAPER_THRESHOLDS[fgiou.argmax()])}

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
LORA_GRAD_CHECKPOINT = False
import math
from contextlib import contextmanager
import torch.utils.checkpoint

class LoRALinear(torch.nn.Module):
    """W·x + (alpha/r)·B·A·x; B khởi tạo 0 nên lúc đầu đúng bằng lớp gốc. Lớp gốc giữ đóng băng."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_a = torch.nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device))
        self.lora_b = torch.nn.Parameter(torch.zeros(base.out_features, rank, device=base.weight.device))
        torch.nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        self.scale = alpha / rank
        self.enabled = True

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + x @ self.lora_a.t().to(x.dtype) @ self.lora_b.t().to(x.dtype) * self.scale
        return out

def lora_layers(model):
    """Các layer CLIP vision có output đi vào decoder (0..max(extract_layers))."""
    return list(model.clip.vision_model.encoder.layers[:max(model.extract_layers) + 1])

def lora_modules(model):
    return [m for m in model.clip.vision_model.modules() if isinstance(m, LoRALinear)]

def has_lora(model):
    return bool(lora_modules(model))

def attach_lora(model, rank, alpha):
    assert not has_lora(model), 'LoRA đã được gắn'
    for layer in lora_layers(model):
        for name in ['q_proj', 'v_proj']:
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name), rank, alpha))

def lora_parameters(model):
    return [p for m in lora_modules(model) for p in (m.lora_a, m.lora_b)]

def trainable_groups(model, lr):
    """Nhóm 0 = decoder (lr), nhóm 1 = LoRA (lr × LORA_LR_SCALE). Không có LoRA thì chỉ một nhóm, như B1."""
    groups = [{'params': list(model.decoder.parameters()), 'lr': lr}]
    if lora_parameters(model):
        groups.append({'params': lora_parameters(model), 'lr': lr * LORA_LR_SCALE})
    return groups

def sync_lora_lr(optimizer):
    """Giữ LR của LoRA đúng bằng LORA_LR_SCALE × LR decoder sau mỗi bước của lịch cosine."""
    if len(optimizer.param_groups) > 1:
        optimizer.param_groups[1]['lr'] = optimizer.param_groups[0]['lr'] * LORA_LR_SCALE

def lora_state_dict(model):
    return {k: v.detach().cpu() for k, v in model.clip.vision_model.state_dict().items() if 'lora_' in k}

def load_lora_state(model, state):
    missing, unexpected = model.clip.vision_model.load_state_dict(state, strict=False)
    assert not unexpected and (not [k for k in missing if 'lora_' in k]), (missing, unexpected)

@contextmanager
def lora_disabled(model):
    """Embedding điều kiện (text, ảnh support) luôn tính bằng CLIP gốc, kể cả ở E4c."""
    modules = lora_modules(model)
    for m in modules:
        m.enabled = False
    try:
        yield
    finally:
        for m in modules:
            m.enabled = True

def vision_activations(model, pixel_values):
    """Output các layer extract_layers của CLIP vision, chỉ chạy tới layer cuối cần dùng."""
    vision = model.clip.vision_model
    hidden = vision.pre_layrnorm(vision.embeddings(pixel_values, interpolate_pos_encoding=True))
    activations = []
    for index, layer in enumerate(lora_layers(model)):
        if LORA_GRAD_CHECKPOINT and torch.is_grad_enabled():
            hidden = torch.utils.checkpoint.checkpoint(layer, hidden, None, None, use_reentrant=False)[0]
        else:
            hidden = layer(hidden, None, None)[0]
        if index in model.extract_layers:
            activations.append(hidden)
    return activations

def mix_conditionals(text_cond, visual_cond, text_weights):
    """cond = w·text + (1−w)·ảnh (giống cell PhraseCut+; định nghĩa sớm để segment dùng được khi chỉ đánh giá)."""
    weights = text_weights.to(text_cond.device, text_cond.dtype)
    return text_cond * weights + visual_cond * (1 - weights)

def segment(model, pixel_values, input_ids, attention_mask, support_pixel_values=None, text_weights=None):
    """Logits [B,352,352] cho train lẫn inference, model gốc (B1) hoặc CLIP vision có LoRA (E4c)."""
    size = pixel_values.shape[0]
    lora = has_lora(model)
    with torch.no_grad():
        if not lora:
            hidden = model.clip.vision_model(pixel_values=pixel_values, output_hidden_states=True, interpolate_pos_encoding=True).hidden_states
            activations = [hidden[i + 1] for i in model.extract_layers]
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=size, input_ids=input_ids, attention_mask=attention_mask)
            if support_pixel_values is not None:
                visual = model.get_conditional_embeddings(batch_size=size, conditional_pixel_values=support_pixel_values)
                cond = mix_conditionals(cond, visual, text_weights)
    if lora:
        activations = vision_activations(model, pixel_values)
    logits = model.decoder(activations, cond).logits
    return logits.reshape(size, *logits.shape[-2:])

def experiment_logits(model, pixel_values, input_ids=None, attention_mask=None, conditional_pixel_values=None):
    """E2/E3 forward: text hoặc ảnh support; với LoRA, embedding điều kiện vẫn dùng CLIP gốc."""
    if not has_lora(model):
        return model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, conditional_pixel_values=conditional_pixel_values).logits
    if (input_ids is None) == (conditional_pixel_values is None):
        raise ValueError('Supply text or support images, not both')
    if input_ids is not None:
        return segment(model, pixel_values, input_ids, attention_mask)
    with torch.no_grad():
        activations = vision_activations(model, pixel_values)
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=pixel_values.shape[0], conditional_pixel_values=conditional_pixel_values)
    return model.decoder(activations, cond).logits


decoder_info = {"source": "pretrained", "checkpoint": None}
if DECODER_CHECKPOINT:
    decoder_path = Path(DECODER_CHECKPOINT)
    if not decoder_path.is_file():
        raise FileNotFoundError(f"DECODER_CHECKPOINT không tồn tại: {decoder_path}")
    checkpoint = torch.load(decoder_path, map_location="cpu", weights_only=True)
    if checkpoint.get("base_model_id") != MODEL_ID or checkpoint.get("base_model_revision") != MODEL_REVISION:
        raise ValueError("Checkpoint được train trên base model/revision khác MODEL_ID/MODEL_REVISION.")
    training = checkpoint.get("training") or {}
    if checkpoint.get("lora"):
        adapter = training.get("lora") or {}
        attach_lora(model, adapter.get("rank", 8), adapter.get("alpha", 16))
        load_lora_state(model, checkpoint["lora"])
    model.decoder.load_state_dict(checkpoint["decoder"])
    model.eval()
    decoder_info = {"source": "e4_trained_decoder" if training.get("arm") else "e1_trained_decoder", "checkpoint": str(decoder_path),
        "checkpoint_sha256": sha256(decoder_path), "epoch": checkpoint["epoch"],
        "global_step": checkpoint["global_step"], "epochs_planned": training.get("epochs"),
        "batch_size": training.get("batch_size"), "initialization": training.get("initialization"),
        "selected_train_queries": training.get("selected_train_queries"),
        # Checkpoint của bản E1 cũ không ghi train_mode: bản đó chỉ train bằng text.
        "train_mode": training.get("train_mode", "text_only" if training else None),
        "history": checkpoint.get("history"), "experiment_arm": training.get("arm"),
        "decoder_arch": training.get("decoder_arch", "clipseg"), "lora": training.get("lora")}
    if training.get("epochs") is not None and checkpoint["epoch"] != training["epochs"]:
        print(f"Cảnh báo: checkpoint epoch {checkpoint['epoch']}/{training['epochs']}, không phải epoch cuối.")
    del checkpoint
print("Decoder:", json.dumps({k: v for k, v in decoder_info.items() if k != "history"}, ensure_ascii=False))

def binary_metrics(probability, target, threshold):
    if probability.shape != target.shape or not target.any():
        raise ValueError("Shape không khớp hoặc GT rỗng.")
    prediction = probability >= threshold
    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    return {"intersection": intersection, "union": union, "iou": intersection / union,
            "dice": 2 * intersection / int(prediction.sum() + target.sum())}

def summarize(rows):
    if not rows:
        raise ValueError("Không có query để tính metric.")
    result = {"n_queries": len(rows), "n_images": len({r["image_id"] for r in rows}),
        "mean_sample_iou": float(np.mean([r["iou"] for r in rows])),
        "cumulative_iou": sum(r["intersection"] for r in rows) / sum(r["union"] for r in rows),
        "mean_dice": float(np.mean([r["dice"] for r in rows]))}
    for cutoff in [0.3, 0.5, 0.7, 0.9]:
        result[f"precision_at_{cutoff}"] = float(np.mean([r["iou"] >= cutoff for r in rows]))
    if "pixel_ap" in rows[0]:
        result["mean_sample_pixel_ap"] = float(np.mean([r["pixel_ap"] for r in rows]))
    return result

@torch.inference_mode()
def predict(image, prompts=None, supports=None):
    """Xác suất [N,H,W] ở kích thước gốc và [N,352,352] để chấm theo paper, cho N prompt trên một ảnh query."""
    if (prompts is None) == (supports is None):
        raise ValueError("Truyền text prompts hoặc support images, không cả hai.")
    count = len(prompts) if prompts is not None else len(supports)
    # Tiền xử lý như CLIPSeg gốc (paper_pixel_values), cho cả ảnh query lẫn ảnh support.
    inputs = {"pixel_values": paper_pixel_values(image).unsqueeze(0).repeat(count, 1, 1, 1)}
    if prompts is not None:
        inputs.update(processor.tokenizer(text=list(prompts), padding=True, truncation=True, return_tensors="pt"))
    else:
        inputs["conditional_pixel_values"] = torch.stack([s if isinstance(s, torch.Tensor) else paper_pixel_values(s)
                                                         for s in supports])
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize()
    started = time.perf_counter()
    context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with context:
        logits = experiment_logits(model, **inputs)
    logits = logits.float().reshape(count, *logits.shape[-2:])
    probabilities = F.interpolate(logits.unsqueeze(1), size=(image.height, image.width),
        mode="bilinear", align_corners=False).squeeze(1).sigmoid()
    synchronize()
    seconds = time.perf_counter() - started
    values = probabilities.cpu().numpy()
    if not np.isfinite(values).all():
        raise ValueError("Prediction chứa NaN/Inf; thử USE_AMP=False.")
    return values, logits.sigmoid().cpu().numpy(), seconds

paper_stats = defaultdict(list)   # biến thể -> tp/fp/fn/tn tại 49 ngưỡng của từng dòng metric

def evaluate(probability, target, base, probability_352):
    """Metric tại THRESHOLD (kèm pixel AP), các dòng quét ngưỡng, và stats theo protocol paper ở 352×352."""
    row = {**base, "threshold": THRESHOLD, **binary_metrics(probability, target, THRESHOLD),
           "pixel_ap": float(average_precision_score(target.ravel(), probability.ravel()))}
    sweep = [{**base, "threshold": t, **binary_metrics(probability, target, t)} for t in THRESHOLD_GRID]
    paper_stats[base["variant"]].append(paper_threshold_stats(probability_352, paper_target(target)))
    return row, sweep

def save_paper_stats():
    np.savez_compressed(OUT / "paper_threshold_stats.npz", thresholds=PAPER_THRESHOLDS,
        **{f"variant_{i}": np.stack(paper_stats[v]) for i, v in enumerate(paper_stats)})
    return {f"variant_{i}": v for i, v in enumerate(paper_stats)}

def variant_report(rows, sweep_rows, stats):
    """Ngưỡng cố định trên mọi query; ngưỡng bổ sung chọn chỉ trên calibration."""
    table = []
    for threshold in THRESHOLD_GRID:
        subset = [r for r in sweep_rows if r["group"] == "calibration" and r["threshold"] == threshold]
        table.append({"threshold": threshold, **summarize(subset)})
    best = max(table, key=lambda r: (r["mean_sample_iou"], -r["threshold"]))["threshold"]
    return {
        "all_fixed_threshold": summarize(rows),
        "calibration_fixed_threshold": summarize([r for r in rows if r["group"] == "calibration"]),
        "evaluation_fixed_threshold": summarize([r for r in rows if r["group"] == "evaluation"]),
        "calibration_selected_threshold": best,
        "evaluation_calibrated_threshold": summarize([r for r in sweep_rows
            if r["group"] == "evaluation" and r["threshold"] == best]),
        "paper_protocol": paper_metrics(stats),
    }, table

def paired_comparison(base_iou, other_iou, image_of, n_bootstrap, tie_tolerance, seed):
    """ΔIoU theo query (other − base), CI 95% bootstrap theo ảnh, số query thắng/hòa/thua."""
    query_ids = sorted(base_iou)
    if set(query_ids) != set(other_iou):
        raise ValueError("Hai biến thể phải có cùng tập query.")
    deltas = np.array([other_iou[q] - base_iou[q] for q in query_ids])
    groups = defaultdict(list)
    for index, query_id in enumerate(query_ids):
        groups[image_of[query_id]].append(index)
    clusters = [np.array(indices) for _, indices in sorted(groups.items())]
    rng = np.random.default_rng(seed)
    boot = [deltas[np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])].mean()
            for _ in range(n_bootstrap)]
    return {"n_queries": len(deltas), "delta_mean_iou": float(deltas.mean()),
            "ci95_low": float(np.percentile(boot, 2.5)), "ci95_high": float(np.percentile(boot, 97.5)),
            "wins": int((deltas > tie_tolerance).sum()), "ties": int((np.abs(deltas) <= tie_tolerance).sum()),
            "losses": int((deltas < -tie_tolerance).sum())}

def overlay(image, probability):
    rgb = np.asarray(image).astype(np.float32).copy()
    mask = probability >= THRESHOLD
    rgb[mask] = 0.55 * rgb[mask] + 0.45 * np.array([255, 60, 60])
    return rgb.astype(np.uint8)

def load_rgb(image_id):
    with Image.open(image_paths[image_id]) as raw:
        return raw.convert("RGB")

def save_mask(probability, variant, stem):
    if SAVE_MASKS:
        folder = OUT / "masks" / variant
        folder.mkdir(parents=True, exist_ok=True)
        Image.fromarray((probability >= THRESHOLD).astype(np.uint8) * 255).save(folder / f"{stem}.png")

def table_row(name, report, comparison):
    fixed = report["all_fixed_threshold"]
    paper = report["paper_protocol"]
    return {"variant": name, "rows": fixed["n_queries"],
        "paper_miou_0.3": paper["miou_0.3"], "paper_fgiou_0.3": paper["fgiou_0.3"], "paper_ap": paper["ap"],
        "mean_iou": fixed["mean_sample_iou"],
        "cumulative_iou": fixed["cumulative_iou"], "mean_dice": fixed["mean_dice"],
        "mean_pixel_ap": fixed["mean_sample_pixel_ap"], "precision_at_0.5": fixed["precision_at_0.5"],
        "calibrated_threshold": report["calibration_selected_threshold"],
        "evaluation_iou_fixed": report["evaluation_fixed_threshold"]["mean_sample_iou"],
        "evaluation_iou_calibrated": report["evaluation_calibrated_threshold"]["mean_sample_iou"],
        **{key: (comparison or {}).get(key) for key in
           ["delta_mean_iou", "ci95_low", "ci95_high", "wins", "ties", "losses"]}}

def print_table(rows):
    columns = ["variant", "rows", "paper_miou_0.3", "paper_fgiou_0.3", "paper_ap", "mean_iou",
               "delta_mean_iou", "ci95_low", "ci95_high", "wins", "losses"]
    print(" | ".join(columns))
    for row in rows:
        print(" | ".join(f"{row[c]:.4f}" if isinstance(row[c], float) else str(row[c]) for c in columns))

def run_metadata(extra):
    inference_seconds = sum(b["forward_resize_seconds"] for b in batch_times)
    prompts = sum(b["prompts"] for b in batch_times)
    return {**extra, "decoder": decoder_info,
        "model_id": MODEL_ID, "requested_revision": MODEL_REVISION if not MODEL_PATH else None,
        "model_source": model_source, "resolved_commit": getattr(model.config, "_commit_hash", None),
        "phrasecut_api_commit": PHRASECUT_API_COMMIT,
        "annotation_sha256": sha256(annotation_path), "image_metadata_sha256": sha256(info_path),
        "seed": SEED, "threshold": THRESHOLD, "threshold_grid": THRESHOLD_GRID,
        "calibration_fraction": CALIBRATION_FRACTION, "n_bootstrap": N_BOOTSTRAP, "tie_tolerance": TIE_TOLERANCE,
        "precision": "fp16_autocast" if USE_AMP else "fp32", "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "torch_cuda_version": torch.version.cuda, "python": platform.python_version(), "versions": versions,
        "image_processor": processor.image_processor.to_dict(),
        "forward_resize_seconds_total": inference_seconds,
        "forward_resize_ms_per_prompt": 1000 * inference_seconds / prompts,
        "wall_seconds_inference_metrics_saving": elapsed_all,
        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None,
        "timing_note": "Two warmup batches; forward+resize+sigmoid timed with CUDA sync; excludes preprocessing and metric/save.",
    }


## 6. Tạo visual prompt và chạy 5 biến thể

`support_views` tạo 3 visual prompt từ ảnh support và mask GT **của ảnh support**.
Text: một forward cho `text_phrase` và `text_name`. Visual: gom (support × 3 biến thể) thành batch `BATCH_SIZE`.
Mask lưu tại `masks/<variant>/<task_id>.png` (text) và `masks/<variant>/<task_id>__<support_task_id>.png` (visual).


In [ ]:
def random_square_crop(target, iterations=50, min_frac=0.05):
    """Crop vuông cạnh min(H, W); nhận crop đầu tiên có số pixel vật > min_frac × diện tích ảnh (find_crop gốc)."""
    height, width = target.shape
    side = min(height, width)
    min_sum = height * width * min_frac
    best = (-1, 0, 0)
    for _ in range(iterations):
        y = int(torch.randint(0, height - side + 1, (1,)))
        x = int(torch.randint(0, width - side + 1, (1,)))
        total = int(target[y:y + side, x:x + side].sum())
        if total > min_sum:
            return (y, x, side)
        if total > best[0]:
            best = (total, y, x)
    return (best[1], best[2], side)

def gaussian_blur(image, kernel_size=15, sigma=3.0):
    """Gaussian blur [C,H,W] biên reflect, tương đương cv2.GaussianBlur(img, (15, 15), 3) của repo gốc."""
    coords = torch.arange(kernel_size, dtype=image.dtype) - (kernel_size - 1) / 2
    kernel = torch.exp(-coords ** 2 / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    channels, pad = (image.shape[0], kernel_size // 2)
    blurred = F.pad(image.unsqueeze(0), (pad, pad, pad, pad), mode='reflect')
    blurred = F.conv2d(blurred, kernel.view(1, 1, 1, -1).repeat(channels, 1, 1, 1), groups=channels)
    blurred = F.conv2d(blurred, kernel.view(1, 1, -1, 1).repeat(channels, 1, 1, 1), groups=channels)
    return blurred.squeeze(0)

def visual_prompt(image, mask, size=352, blur_sigma=3.0, background_factor=0.1, context=0.1):
    """crop_blur_highlight352: nền làm mờ còn 10% (trên ảnh đã chuẩn hóa), crop quanh vật nới 10%, pad vuông, resize."""
    mask = mask.to(image.dtype)
    blended = image * mask + background_factor * gaussian_blur(image, sigma=blur_sigma) * (1 - mask)
    ys, xs = torch.nonzero(mask > 0, as_tuple=True)
    if len(ys):
        y0, y1, x0, x1 = (int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1)
    else:
        y0, y1, x0, x1 = (0, mask.shape[0], 0, mask.shape[1])
    dy, dx = (context * (y1 - y0), context * (x1 - x0))
    y0, y1 = (int(max(0, y0 - dy)), int(min(mask.shape[0], y1 + dy)))
    x0, x1 = (int(max(0, x0 - dx)), int(min(mask.shape[1], x1 + dx)))
    crop = blended[:, y0:y1, x0:x1]
    height, width = crop.shape[1:]
    side = max(height, width)
    top, left = ((side - height) // 2, (side - width) // 2)
    crop = F.pad(crop, (left, side - width - left, top, side - height - top))
    return F.interpolate(crop.unsqueeze(0), size=(size, size), mode='nearest').squeeze(0)

TEXT_VARIANTS = ["text_phrase", "text_name"]
assert SUPPORT_VIEW_MODE in ["legacy", "train_matched"]
VISUAL_VARIANTS = ["visual_crop", "visual_dark", "visual_crop_dark"] if SUPPORT_VIEW_MODE == "legacy" else ["visual_train"]
VARIANTS = TEXT_VARIANTS + VISUAL_VARIANTS
BASELINE_VARIANT = "text_phrase"
assert BATCH_SIZE >= 1
if decoder_info.get("train_mode") == "text_only":
    print("CẢNH BÁO: decoder chỉ được train bằng text, chưa từng thấy embedding ảnh support. "
          "Kết quả visual chỉ dùng làm ablation, không so ngang với decoder train PhraseCut+.")
(OUT / "visualizations").mkdir(exist_ok=True)

def support_views(image, mask, crop_padding, dark_factor):
    """Crop / tối nền / kết hợp, chỉ dùng mask của chính ảnh support."""
    ys, xs = np.nonzero(mask)
    x0, x1, y0, y1 = int(xs.min()), int(xs.max()) + 1, int(ys.min()), int(ys.max()) + 1
    pad_x, pad_y = round((x1 - x0) * crop_padding), round((y1 - y0) * crop_padding)
    box = (max(0, x0 - pad_x), max(0, y0 - pad_y), min(image.width, x1 + pad_x), min(image.height, y1 + pad_y))
    rgb = np.asarray(image, dtype=np.float32)
    dark = Image.fromarray(np.where(mask[..., None], rgb, rgb * dark_factor).round().astype(np.uint8))
    return {"visual_crop": image.crop(box), "visual_dark": dark, "visual_crop_dark": dark.crop(box)}

def training_support_seed(task_id):
    return int(hashlib.sha256(f"{SEED}:{task_id}".encode()).hexdigest()[:8], 16)

def training_support(image, mask, seed=None):
    # Same operations as load_train_sample + visual_prompt, with a fixed per-support crop draw.
    pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1)
    target = mask
    if SUPPORT_AUG_CROP:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(SEED if seed is None else seed)
            y, x, side = random_square_crop(target)
        pixels, target = pixels[:, y:y + side, x:x + side], target[y:y + side, x:x + side]
    pixels = F.interpolate(pixels.unsqueeze(0), size=(352, 352), mode="bilinear", align_corners=True)[0]
    target = F.interpolate(torch.from_numpy(target.astype(np.float32))[None, None],
                           size=(352, 352), mode="nearest")[0, 0]
    pixels = (pixels / 255 - image_mean) / image_std
    return visual_prompt(pixels, target, background_factor=DARK_FACTOR, context=CROP_PADDING)

def support_preview(value):
    if isinstance(value, torch.Tensor):
        rgb = value * image_std + image_mean
        return rgb.permute(1, 2, 0).clamp(0, 1).numpy()
    return value

views_cache = {}
def views_for(support):
    key = str(support["task_id"])
    if key not in views_cache:
        image = load_rgb(int(support["image_id"]))
        mask = rasterize_gt(support, image.width, image.height)
        views_cache[key] = (support_views(image, mask, CROP_PADDING, DARK_FACTOR) if SUPPORT_VIEW_MODE == "legacy"
                            else {"visual_train": training_support(image, mask, training_support_seed(key))})
    return views_cache[key]

def save_pair_figure(image, target, pair, probabilities, ious, index):
    task, support = pair["query"], pair["supports"][0]
    panels = [("Ảnh query", image), ("Ground truth", target),
              (f"text_phrase IoU={ious['text_phrase']:.3f}", overlay(image, probabilities["text_phrase"])),
              (f"Support: {support['phrase']}", support_preview(views_for(support)[VISUAL_VARIANTS[-1]]))]
    panels += [(f"{v} IoU={ious[v]:.3f}", overlay(image, probabilities[v])) for v in VISUAL_VARIANTS]
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    for ax, (title, value) in zip(axes, panels):
        ax.imshow(value, cmap="gray" if title == "Ground truth" else None)
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{task['task_id']} — {task['phrase']} (visual dùng support đầu tiên)")
    fig.tight_layout()
    fig.savefig(OUT / "visualizations" / f"{index:02d}_{task['task_id']}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

warmup_image = load_rgb(int(pairs[0]["query"]["image_id"]))
for _ in range(2):
    predict(warmup_image, prompts=[pairs[0]["query"]["phrase"], target_key(pairs[0]["query"])])
    predict(warmup_image, supports=[views_for(pairs[0]["supports"][0])[v] for v in VISUAL_VARIANTS])
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

rows, sweep_rows, batch_times = [], [], []
started_all = time.perf_counter()
for index, pair in enumerate(tqdm(pairs, desc="E2 queries")):
    task = pair["query"]
    task_id, image_id = str(task["task_id"]), int(task["image_id"])
    image = load_rgb(image_id)
    target = rasterize_gt(task, image.width, image.height)
    base = {"task_id": task_id, "image_id": image_id, "phrase": task["phrase"], "target_key": target_key(task),
            "group": group_for_image[image_id], "width": image.width, "height": image.height}
    first_probabilities, first_ious = {}, {}
    text_prompts = {"text_phrase": task["phrase"], "text_name": target_key(task)}
    probabilities, probabilities_352, seconds = predict(image, prompts=list(text_prompts.values()))
    batch_times.append({"prompts": len(text_prompts), "forward_resize_seconds": seconds})
    for (variant, prompt), probability, probability_352 in zip(text_prompts.items(), probabilities, probabilities_352):
        row, sweep = evaluate(probability, target, {**base, "variant": variant, "prompt": prompt,
                                                    "support_task_id": "", "support_image_id": ""}, probability_352)
        rows.append(row)
        sweep_rows.extend(sweep)
        save_mask(probability, variant, task_id)
        first_probabilities[variant], first_ious[variant] = probability, row["iou"]
    jobs = [(support, variant) for support in pair["supports"] for variant in VISUAL_VARIANTS]
    for start in range(0, len(jobs), BATCH_SIZE):
        chunk = jobs[start:start + BATCH_SIZE]
        probabilities, probabilities_352, seconds = predict(image, supports=[views_for(s)[v] for s, v in chunk])
        batch_times.append({"prompts": len(chunk), "forward_resize_seconds": seconds})
        for (support, variant), probability, probability_352 in zip(chunk, probabilities, probabilities_352):
            row, sweep = evaluate(probability, target, {**base, "variant": variant,
                "prompt": f"support:{support['task_id']}", "support_task_id": str(support["task_id"]),
                "support_image_id": int(support["image_id"])}, probability_352)
            rows.append(row)
            sweep_rows.extend(sweep)
            save_mask(probability, variant, f"{task_id}__{support['task_id']}")
            if support is pair["supports"][0]:
                first_probabilities[variant], first_ious[variant] = probability, row["iou"]
    if index < N_VISUALIZATIONS:
        save_pair_figure(image, target, pair, first_probabilities, first_ious, index)
    write_csv(OUT / "sample_metrics.csv", rows)
elapsed_all = time.perf_counter() - started_all
assert len(rows) == len(pairs) * (len(TEXT_VARIANTS) + len(VISUAL_VARIANTS) * SUPPORTS_PER_QUERY)
print(f"Completed {len(pairs)} queries × {len(VARIANTS)} variants; wall time {elapsed_all:.1f}s")


## 7. Bảng E2

`e2_variant_table.csv` là bảng chính. Cột `rows` là số dòng metric: N query với text,
N × K (query × support) với visual; mean IoU, cumulative IoU, AP và precision tính trên các dòng đó.
ΔIoU so với `text_phrase` tính **theo query**: IoU visual được trung bình trên K support trước khi trừ.
CI 95% bootstrap theo ảnh query. Visual prompt thường cần ngưỡng khác text, nên xem thêm cột
calibrated (ngưỡng chọn trên calibration) và AP (không phụ thuộc ngưỡng).


In [ ]:
image_of = {str(t["task_id"]): int(t["image_id"]) for t in selected}
reports, calibration_rows, table = {}, [], []
iou_by_variant = {}
for variant in VARIANTS:
    variant_rows = [r for r in rows if r["variant"] == variant]
    report, calibration_table = variant_report(variant_rows, [r for r in sweep_rows if r["variant"] == variant],
                                               paper_stats[variant])
    reports[variant] = report
    calibration_rows += [{"variant": variant, **r} for r in calibration_table]
    per_query = defaultdict(list)
    for r in variant_rows:
        per_query[r["task_id"]].append(r["iou"])
    iou_by_variant[variant] = {task_id: float(np.mean(values)) for task_id, values in per_query.items()}
    reports[variant]["mean_iou_per_query"] = float(np.mean(list(iou_by_variant[variant].values())))
for variant in VARIANTS:
    comparison = None if variant == BASELINE_VARIANT else paired_comparison(
        iou_by_variant[BASELINE_VARIANT], iou_by_variant[variant], image_of, N_BOOTSTRAP, TIE_TOLERANCE, SEED)
    reports[variant]["paired_vs_baseline"] = comparison
    table.append(table_row(variant, reports[variant], comparison))
write_csv(OUT / "calibration_thresholds.csv", calibration_rows)
write_csv(OUT / "e2_variant_table.csv", table)
paper_stats_keys = save_paper_stats()
summary = {"experiment": "E2_text_vs_visual", "decoder": decoder_info, "dataset": "VGPhraseCut", "split": EVAL_SPLIT,
    "n_queries": len(pairs), "supports_per_query": SUPPORTS_PER_QUERY, "baseline_variant": BASELINE_VARIANT,
    "variants": reports,
    "paper_threshold_stats_keys": paper_stats_keys,
    "metric_note": ("paper_protocol per variant follows CLIPSeg score.py/FixedIntervalMetrics at 352x352 "
                    "(visual rows pooled over query×support). Text rows: one per query; visual rows: one per query×support. "
                    "ΔIoU is per query after averaging visual IoU over supports; CI is image-level bootstrap.")}
write_json(OUT / "summary.json", summary)
write_json(OUT / "run.json", run_metadata({"experiment": "E2_text_vs_visual", "max_queries": MAX_QUERIES,
    "supports_per_query": SUPPORTS_PER_QUERY, "min_support_box_fraction": MIN_SUPPORT_BOX_FRACTION,
    "crop_padding": CROP_PADDING, "dark_factor": DARK_FACTOR, "support_view_mode": SUPPORT_VIEW_MODE,
    "support_aug_crop": SUPPORT_AUG_CROP if SUPPORT_VIEW_MODE == "train_matched" else False,
    "variants": VARIANTS, "batch_size": BATCH_SIZE,
    "pairs_sha256": sha256(OUT / "e2_pairs.json"),
    "calibration_image_ids": sorted(calibration_ids),
    "evaluation_image_ids": sorted(set(query_image_ids) - calibration_ids)}))
print_table(table)


## 8. Xem hình và đóng gói

Tải `clipseg_e2_text_vs_visual.zip` trong Kaggle Output.


In [ ]:
for path in sorted((OUT / "visualizations").glob("*.png"))[:N_VISUALIZATIONS]:
    with Image.open(path) as image:
        display(image.copy())
archive = WORK_ROOT / f"{OUTPUT_NAME}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            package.write(path, arcname=path.relative_to(OUT).as_posix())
print("Result folder:", OUT)
print("Download:", archive)


## Nguồn và giới hạn

- [CLIPSeg / CVPR 2022](https://github.com/timojl/clipseg), [checkpoint HF](https://huggingface.co/CIDAS/clipseg-rd64-refined).
- [PhraseCutDataset](https://github.com/ChenyunWu/PhraseCutDataset): annotation, URL ảnh, split và quy ước polygon/mask.
- [CLIPSeg trong Transformers 4.55.4](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg).

Checkpoint pretrained, không train. PhraseCut liên quan tới dữ liệu huấn luyện CLIPSeg nên không gọi là zero-shot.
Google Drive/Visual Genome có thể gián đoạn; Add Input dataset chứa hai file JSON (và ảnh) là đường dự phòng.
